In [1]:
import numpy as np

1) Reshape round-trip. Take a tensor of shape (2, 3, 4). Reshape it to (6, 4), then to (24,), then back to (2, 3, 4). Verify element order is preserved at each step by printing the flat data.

In [2]:
import numpy as np

A = np.arange(24).reshape(2, 3, 4)

print("Original Shape:", A.shape)
print(A)

B = A.reshape(6, 4)
print("\nReshaped to (6, 4):")
print(B)

C = B.reshape(24,)
print("\nReshaped to (24,):")
print(C)

D = C.reshape(2, 3, 4)
print("\nBack to (2, 3, 4):")
print(D)

print("\nOriginal Flat Data:")
print(A.flatten())

print("\nFinal Flat Data:")
print(D.flatten())

print("\nElement Order Preserved:", np.array_equal(A, D))

Original Shape: (2, 3, 4)
[[[ 0  1  2  3]
  [ 4  5  6  7]
  [ 8  9 10 11]]

 [[12 13 14 15]
  [16 17 18 19]
  [20 21 22 23]]]

Reshaped to (6, 4):
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]
 [16 17 18 19]
 [20 21 22 23]]

Reshaped to (24,):
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]

Back to (2, 3, 4):
[[[ 0  1  2  3]
  [ 4  5  6  7]
  [ 8  9 10 11]]

 [[12 13 14 15]
  [16 17 18 19]
  [20 21 22 23]]]

Original Flat Data:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]

Final Flat Data:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]

Element Order Preserved: True


3) Implement broadcasting. Extend the Tensor class with a broadcast_to(shape) method that expands dimensions of size 1 to match a target shape. Then modify _elementwise_op to automatically broadcast before operating. Test with shapes (3, 1) and (1, 4) producing (3, 4).

In [3]:
import numpy as np

def broadcast_to(A, shape):
    return np.broadcast_to(A, shape).copy()


def elementwise_op(A, B, operation):

    A = np.array(A)
    B = np.array(B)

    shape = np.broadcast_shapes(A.shape, B.shape)

    A = broadcast_to(A, shape)
    B = broadcast_to(B, shape)

    return operation(A, B)


A = np.array([
    [1],
    [2],
    [3]
])

B = np.array([
    [10, 20, 30, 40]
])

result = elementwise_op(A, B, np.add)

print("A Shape:", A.shape)
print("B Shape:", B.shape)

print("\nResult:")
print(result)

print("\nResult Shape:", result.shape)

A Shape: (3, 1)
B Shape: (1, 4)

Result:
[[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]

Result Shape: (3, 4)


3) Build einsum from scratch. Implement a basic einsum(subscripts, *tensors) function that handles at least: dot product (i,i->), matrix multiply (ij,jk->ik), outer product (i,j->ij), and transpose (ij->ji). Parse the subscript string, identify contracted indices, and loop over all index combinations. Compare your results against np.einsum.

In [4]:
import numpy as np
import itertools

def einsum(subscripts, *tensors):

    inputs, output = subscripts.split("->")
    inputs = inputs.split(",")

    shapes = [t.shape for t in tensors]

    all_indices = "".join(inputs)
    unique_indices = set(all_indices)

    dimensions = {}

    for i, tensor in enumerate(tensors):
        for j, index in enumerate(inputs[i]):
            dimensions[index] = tensor.shape[j]

    result_shape = tuple(dimensions[i] for i in output)

    result = np.zeros(result_shape)

    for values in itertools.product(
        *[range(dimensions[i]) for i in unique_indices]
    ):

        index_values = dict(zip(unique_indices, values))

        product = 1

        for tensor, indices in zip(tensors, inputs):

            tensor_index = tuple(index_values[i] for i in indices)

            product *= tensor[tensor_index]

        if output:
            output_index = tuple(index_values[i] for i in output)
            result[output_index] += product

        else:
            result += product

    return result

In [5]:
# 1. Dot product
A = np.array([1, 2, 3])
B = np.array([4, 5, 6])

result1 = einsum("i,i->", A, B)
expected1 = np.einsum("i,i->", A, B)

print("Dot Product:")
print(result1)
print("NumPy:", expected1)


# 2. Matrix multiplication
A = np.array([
    [1, 2],
    [3, 4]
])

B = np.array([
    [5, 6],
    [7, 8]
])

result2 = einsum("ij,jk->ik", A, B)
expected2 = np.einsum("ij,jk->ik", A, B)

print("\nMatrix Multiplication:")
print(result2)
print("NumPy:")
print(expected2)


# 3. Outer product
A = np.array([1, 2])
B = np.array([3, 4, 5])

result3 = einsum("i,j->ij", A, B)
expected3 = np.einsum("i,j->ij", A, B)

print("\nOuter Product:")
print(result3)
print("NumPy:")
print(expected3)


# 4. Transpose
A = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

result4 = einsum("ij->ji", A)
expected4 = np.einsum("ij->ji", A)

print("\nTranspose:")
print(result4)
print("NumPy:")
print(expected4)


print("\nVerification:")
print(np.allclose(result1, expected1))
print(np.allclose(result2, expected2))
print(np.allclose(result3, expected3))
print(np.allclose(result4, expected4))

Dot Product:
32.0
NumPy: 32

Matrix Multiplication:
[[19. 22.]
 [43. 50.]]
NumPy:
[[19 22]
 [43 50]]

Outer Product:
[[ 3.  4.  5.]
 [ 6.  8. 10.]]
NumPy:
[[ 3  4  5]
 [ 6  8 10]]

Transpose:
[[1. 4.]
 [2. 5.]
 [3. 6.]]
NumPy:
[[1 4]
 [2 5]
 [3 6]]

Verification:
True
True
True
True
